# Phase 2 matched experiment: implementation

This notebook runs only the approved `phase3a_stable_cohort` experiment using the shared `scripts/shared_monitoring_engine.py` engine. It publishes the summary under `artifacts/phase2_outputs/` and event evidence under `artifacts/phase2_operational/`. Run the test notebook afterward to issue the independent validation receipt.

Run every cell from a clean kernel. Missing or inconsistent inputs, failed correctness/accounting checks, and export mismatches raise exceptions and stop execution.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "config" / "project_config.yaml").is_file():
    parent = REPO_ROOT.parent
    if parent == REPO_ROOT:
        raise FileNotFoundError("Run this notebook from the sensor_project repository.")
    REPO_ROOT = parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.shared_monitoring_engine import POLICY_ID, load_phase3a_config, publish

CONFIG_PATH = REPO_ROOT / "config" / "project_config.yaml"
INPUT_DIR = REPO_ROOT / "artifacts"
SUMMARY_PATH = INPUT_DIR / "phase2_outputs" / "phase2_summary.json"
EVENTS_PATH = INPUT_DIR / "phase2_operational" / "phase2_events.csv"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print(f"Repository: {REPO_ROOT}")
print(f"Shared monitoring policy: {POLICY_ID}")

## 1. Validate inputs and resolve the approved policy

The configuration locks the cohort, inclusive interval, cadence, source files, and threshold identity. Derive the full-precision cohort P90 through the shared engine; its evidence is saved in the result, not configuration. The engine repeats these gates before execution, including the exact bucket timestamps, epochs, cadence, complete finite sensor readings, and historical-reference provenance.

In [ ]:
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f"Missing experiment configuration: {CONFIG_PATH}")
config = load_phase3a_config(REPO_ROOT)
threshold_config = config["threshold"]
input_path = REPO_ROOT / config["input_source"]
from scripts.shared_monitoring_engine import prepare_monitoring
monitoring_preparation = prepare_monitoring(REPO_ROOT)
threshold = monitoring_preparation["p90_value"]
if config["experiment_name"] != "phase3a_stable_cohort":
    raise ValueError("Only the approved phase3a_stable_cohort experiment may run here.")

print(json.dumps({
    "experiment": config["experiment_name"],
    "sensors": config["selected_sensor_ids"],
    "start_utc_inclusive": config["start_timestamp_utc"],
    "end_utc_inclusive": config["end_timestamp_utc"],
    "expected_buckets": config["expected_bucket_count"],
    "threshold_policy": POLICY_ID,
    "threshold_series": threshold_config["source_series"],
    "threshold_p90": threshold,
}, indent=2))

## 2. Run the shared monitoring engine

All state transitions, unrounded monitoring decisions, independent root-state correctness checks, and transmission accounting are implemented in `shared_monitoring_engine.py`. This notebook does not contain a second state machine and does not execute the full-dataset workflow.

In [ ]:
summary = publish(REPO_ROOT)
if summary["correctness_status"] != "passed":
    raise RuntimeError(f"Independent root-state correctness gate failed: {summary}")
if summary["message_counts_reconcile"] is not True:
    raise RuntimeError("Phase 2 message categories do not reconcile to total transmissions.")
if summary["root_state_evaluation_count"] != config["expected_bucket_count"]:
    raise RuntimeError("Not every matched bucket received an independent root-state check.")
if summary["false_safe_count"] or summary["false_violated_count"] or summary["root_state_not_evaluable_count"]:
    raise RuntimeError("The matched root-state correctness gate reported errors.")
if summary["total_messages"] != sum(summary["message_category_totals"].values()):
    raise RuntimeError("Phase 2 category totals do not equal total transmissions.")

display(pd.DataFrame([
    {"measure": "Matched buckets", "value": summary["bucket_count"]},
    {"measure": "Threshold violations", "value": summary["global_threshold_violation_count"]},
    {"measure": "Root synchronizations", "value": summary["resynchronization_count"]},
    {"measure": "Total transmissions", "value": summary["total_messages"]},
    {"measure": "False-safe root states", "value": summary["false_safe_count"]},
    {"measure": "False-violated root states", "value": summary["false_violated_count"]},
    *[
        {"measure": name.replace("_message_count", " transmissions"), "value": count}
        for name, count in summary["message_category_totals"].items()
    ],
]))

## 3. Verify current exports

The baseline summary and completed-synchronization event table remain the existing Phase 2 outputs. The summary references the exact event-file hash. Any missing, stale, partially written, or unreconciled output fails the notebook.

In [ ]:
if not SUMMARY_PATH.is_file() or not EVENTS_PATH.is_file():
    raise FileNotFoundError("The matched Phase 2 summary or event export is missing.")
saved_summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
events = pd.read_csv(EVENTS_PATH, keep_default_na=False)
if saved_summary != summary:
    raise ValueError("The saved matched summary differs from the completed engine run.")
if sha256(EVENTS_PATH) != summary["events_sha256"]:
    raise ValueError("The saved matched event file does not match the summary hash.")
if len(events) != summary["resynchronization_count"]:
    raise ValueError("Event rows do not reconcile to completed synchronizations.")
if int(events["total_message_count"].sum()) != summary["total_messages"]:
    raise ValueError("Event-level transmissions do not reconcile to the summary.")
if set(events["root_known_state"]) - {"SAFE", "VIOLATED"}:
    raise ValueError("A completed synchronization has an invalid root-known state.")

print(f"Summary: {SUMMARY_PATH.relative_to(REPO_ROOT)}")
print(f"Events: {EVENTS_PATH.relative_to(REPO_ROOT)} ({len(events):,} rows)")
print(f"Event SHA-256: {summary['events_sha256']}")
print("Matched Phase 2 implementation and export checks passed.")